# 国风炼金卡牌 · 云端批量图生图
## Google Colab + SDXL + ControlNet

**使用方法**：
1. 把参考图打包成 `ref_images.zip` 上传到你的 Google Drive 根目录
2. 点击菜单栏 "代码执行程序" → "全部运行"
3. 生成的卡牌会自动保存到 Google Drive 的 `card_output/` 文件夹

**硬件**：T4 GPU 16GB 免费

In [ ]:
# 1. 安装依赖
!pip install diffusers transformers accelerate xformers pillow controlnet-aux -q

In [ ]:
# 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
from pathlib import Path
import torch
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
from diffusers.utils import load_image
import numpy as np

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# 3. 加载模型（国风SDXL + ControlNet Canny）
# 使用中国风 LoRA 模型

model_id = "RunDiffusion/Juggernaut-XL-v9"  # 写实模型作为基底

# ControlNet Canny
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=torch.float16
)

# VAE
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
)

# 主模型
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    model_id,
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
)

pipe = pipe.to("cuda")
pipe.enable_xformers_memory_efficient_attention()

print("模型加载完成！")

In [ ]:
# 4. 解压参考图（如果上传了ref_images.zip）
import zipfile, shutil

ref_dir = Path("/content/ref_images")
zip_path = Path("/content/drive/MyDrive/ref_images.zip")

if zip_path.exists():
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ref_dir)
    print(f"已解压 {len(list(ref_dir.glob('*')))} 个文件")
else:
    ref_dir.mkdir(exist_ok=True)
    print("未找到 ref_images.zip，使用演示模式生成3张测试")
    
    # 演示：从网上下载几张历史参考图
    test_cards = [
        ("秦始皇", "Qin Shi Huang, first emperor, ancient Chinese portrait, historical figure"),
        ("关羽", "Guan Yu, Three Kingdoms general, Chinese warrior portrait"),
        ("咸阳", "Xianyang ancient city, Qin dynasty capital, Chinese architecture"),
    ]
    # 生成简单参考图
    for name, prompt in test_cards:
        # 用一个小模型快速生成参考图
        ref_img = Image.new('RGB', (832, 1216), (80, 60, 40))
        ref_img.save(ref_dir / f"{name}.jpg")
        print(f"  参考图: {name}.jpg")

In [ ]:
# 5. 辅助函数：Canny边缘检测
from controlnet_aux import CannyDetector
canny = CannyDetector()

def process_ref(img_path):
    """加载参考图并提取Canny边缘"""
    img = Image.open(img_path).convert("RGB")
    img = img.resize((832, 1216))
    edge = canny(img, low_threshold=80, high_threshold=180)
    return img, edge

# 卡牌提示词模板
PROMPTS = {
    "人物": "masterpiece, best quality, epic Chinese historical portrait of {name}, masculine mature man, stern dignified face, short beard, ancient Chinese Hanfu clothing, dramatic lighting, historical drama style, highly detailed, 2d illustration",
    "事件": "masterpiece, best quality, epic Chinese historical battle scene of {name}, ancient Chinese armies, warriors, dramatic conflict, cinematic wide shot, historical painting style, highly detailed, 2d illustration",
    "地点": "masterpiece, best quality, ancient Chinese landmark {name}, grand architecture, palaces, city walls, misty mountains, cinematic landscape, no humans, historical painting style, 2d illustration",
    "兵器": "masterpiece, best quality, ancient Chinese weapon {name}, ornate metalwork, displayed on silk, museum quality, no humans, still life, historical style, 2d illustration",
    "典籍": "masterpiece, best quality, ancient Chinese classic {name}, bamboo slips, silk scrolls, ink brush, calligraphy, warm candlelight, no humans, still life, historical style, 2d illustration",
}

NEG = "1girl, female, feminine, anime, manga, cartoon, chibi, bishounen, pretty boy, soft face, big eyes, makeup, nsfw, nude, lowres, worst quality, blurry, deformed, modern, gun, western, text, photo, realistic"

print("准备就绪！")

In [ ]:
# 6. 批量生成！
output_dir = Path("/content/drive/MyDrive/card_output")
output_dir.mkdir(parents=True, exist_ok=True)

ref_files = sorted(ref_dir.glob("*.jpg")) + sorted(ref_dir.glob("*.png"))
print(f"共 {len(ref_files)} 张参考图")

for i, rf in enumerate(ref_files):
    name = rf.stem
    print(f"\n[{i+1}/{len(ref_files)}] {name}")
    
    # 加载参考图 + Canny
    ref_img, edge_img = process_ref(rf)
    
    # 选择提示词类型（根据文件名判断，默认人物）
    if any(k in name for k in ['战','宴','变']):
        ptype = "事件"
    elif any(k in name for k in ['城','壁','关','阳','山']):
        ptype = "地点"
    elif any(k in name for k in ['剑','刀','枪','弓','弩']):
        ptype = "兵器"
    elif any(k in name for k in ['经','书','典','术','法']):
        ptype = "典籍"
    else:
        ptype = "人物"
    
    prompt = PROMPTS[ptype].format(name=name)
    
    # 图生图
    result = pipe(
        prompt=prompt,
        negative_prompt=NEG,
        image=edge_img,
        controlnet_conditioning_scale=0.7,
        num_inference_steps=28,
        guidance_scale=6.5,
        width=832,
        height=1216,
    ).images[0]
    
    # 保存
    out_path = output_dir / f"{name}.png"
    result.save(out_path)
    print(f"  ✅ -> {out_path}")

In [ ]:
# 7. 完成！检查结果
outputs = list(output_dir.glob("*.png"))
print(f"\n🎉 生成完成！共 {len(outputs)} 张卡牌")
print(f"保存在: Google Drive/card_output/")

# 显示几张预览
from IPython.display import display
for out in outputs[:5]:
    img = Image.open(out)
    display(img.resize((260, 380)))